## 1. Import required Libraries

In [3]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

SEED = 983
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

KeyboardInterrupt: 

## 2. Import data

In [ ]:
curr_path = Path.cwd()
Output_Datasets_path = curr_path.parent.parent / 'Output_Datasets'

In [ ]:
X_train_clean_path = Output_Datasets_path / 'X_train_clean.npy'
Y_train_clean_path = Output_Datasets_path / 'Y_train_clean.npy'
X_val_clean_path = Output_Datasets_path / 'X_val_clean.npy'
Y_val_clean_path = Output_Datasets_path / 'Y_val_clean.npy'

In [ ]:
print("Loading X_train_clean...")
X_train_clean = np.load(X_train_clean_path, allow_pickle=True)
print("Loading Y_train_clean...")
Y_train_clean = np.load(Y_train_clean_path, allow_pickle=True)
print("Successfully loaded X_train_clean and Y_train_clean")

Loading X_train_clean...
Loading Y_train_clean...
Successfully loaded X_train_clean and Y_train_clean


In [ ]:
print("Loading X_val_clean...")
X_val_clean = np.load(X_val_clean_path, allow_pickle=True)
print("Loading Y_val_clean...")
Y_val_clean = np.load(Y_val_clean_path, allow_pickle=True)
print("Successfully loaded X_val_clean and Y_val_clean")

Loading X_val_clean...
Loading Y_val_clean...
Successfully loaded X_val_clean and Y_val_clean


In [ ]:
print(f'Shape of X_train_clean: {X_train_clean.shape}')
print(f'Shape of Y_train_clean: {Y_train_clean.shape}')

Shape of X_train_clean: (17090, 1000, 12)
Shape of Y_train_clean: (17090, 5)


In [ ]:
print(f'Shape of X_val_clean: {X_val_clean.shape}')
print(f'Shape of Y_val_clean: {Y_val_clean.shape}')

Shape of X_val_clean: (17090, 1000, 12)
Shape of Y_val_clean: (17090, 5)


In [ ]:
X_train_clean = X_train_clean[:1800, :, :]
Y_train_clean = Y_train_clean[:1800, :]
print(f'Shape of X_train_clean: {X_train_clean.shape}')
print(f'Shape of Y_train_clean: {Y_train_clean.shape}')

Shape of X_train_clean: (1800, 1000, 12)
Shape of Y_train_clean: (1800, 5)


In [ ]:
X_val_clean = X_val_clean[:8000, :, :]
Y_val_clean = Y_val_clean[:8000, :]
print(f'Shape of X_val_clean: {X_val_clean.shape}')
print(f'Shape of Y_val_clean: {Y_val_clean.shape}')

Shape of X_val_clean: (8000, 1000, 12)
Shape of Y_val_clean: (8000, 5)


In [ ]:
print(Y_train_clean[0])
print(Y_train_clean[290])

[0 0 0 1 0]
[1 0 0 0 0]


In [ ]:
X_train_clean[0].shape

(1000, 12)

In [ ]:
torch.tensor(X_train_clean[0,:,0]).shape

torch.Size([1000])

## 3. PyTorch DataSet 

#### [BATCH_SIZE, SEQUENCE_LENGHT, INPUT_SIZE] -> [BATCH_SIZE, SEQUENCE_LENGTH 800, INPUT_SIZE 1 TO EMBEDDING_DIM]

In [ ]:
COLUMN = 0

In [ ]:
class EcgDataset(Dataset):
    def __init__(self, inputs, target, column=0):
        self.inputs = inputs
        self.target = target
        self.column = column

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx, :, self.column], dtype=torch.float32), torch.tensor(self.target[idx], dtype=torch.float32) # imp mistake: I tried to give 2d index for a 3d vector

In [ ]:
print('Preparing dataset...')
train_dataset = EcgDataset(inputs=X_train_clean, target=Y_train_clean)
val_dataset = EcgDataset(X_val_clean, Y_val_clean)
print('Completed dataset preparation!')

Preparing dataset...
Completed dataset preparation!


## 4. PyTorch Dataloader

In [ ]:
BATCH_SIZE = 30

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

NameError: name 'DataLoader' is not defined

In [ ]:
print(len(train_dataloader))

60


In [ ]:
for i, x in enumerate(train_dataloader):
    # if i == 2:
    #     print(x[2].shape)
    #     break
    # print(x[0])
    # print(x[1])
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59


In [ ]:
for i, x in enumerate(train_dataloader):
    if i == 2:
        break
    print(x[0].shape)
    print(x[0][0].shape)
    print(x[0])
    print(x[1])
    

torch.Size([30, 1000])
torch.Size([1000])
tensor([[ 6.9461e-01,  6.9461e-01,  6.9461e-01,  ..., -6.3848e-02,
         -2.3792e-01, -3.0009e-01],
        [ 9.1842e-01,  1.0552e+00,  1.2293e+00,  ..., -7.0419e-01,
         -1.1358e-01, -4.5551e-01],
        [ 5.5784e-01,  3.7755e-01,  3.4646e-01,  ..., -5.1146e-01,
          9.1573e-02, -1.8197e-01],
        ...,
        [ 3.5621e-02, -1.6797e-03, -6.3848e-02,  ...,  5.3297e-01,
          4.1485e-01,  1.9104e-01],
        [ 4.2107e-01,  3.8998e-01,  3.5890e-01,  ..., -3.9334e-01,
         -4.9903e-01, -4.3686e-01],
        [ 3.9522e+00,  2.4602e+00,  5.8271e-01,  ..., -1.3880e+00,
         -1.1456e+00, -8.9691e-01]])
tensor([[0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.],
        [1., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.],
        [0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1

## 4. Bidirectional LSTM Model

In [ ]:
SIGNAL_SIZE = 1 # 1 feature per time step (single lead). Change to 12 if passing all 12 leads.
NUM_LAYERS = 2
OUTPUT_DIM = 5
HIDDEN_DIM = 90
EMBED_DIM = 40
DROPOUT = 0.2

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=HIDDEN_DIM, embed_dim=EMBED_DIM, output_dim=OUTPUT_DIM, num_layers=NUM_LAYERS, dropout=0.25):
        super(BiLSTM, self).__init__()

        # Projects each time step's feature(s) into embed_dim
        self.proj = nn.Linear(input_dim, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bias=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, signal):
        # Ensure 3D shape: [batch_size, seq_len, input_dim]
        if signal.ndim == 2:
            signal = signal.unsqueeze(-1)  # [batch_size, 800] -> [batch_size, 800, 1]
            
        # [batch_size, seq_len, 1] -> [batch_size, seq_len, embed_dim]
        projected = self.dropout(torch.relu(self.proj(signal)))
        
        # lstm_out: [batch_size, seq_len, hidden_dim*2]
        # hidden:   [num_layers*2, batch_size, hidden_dim]
        lstm_out, (hidden, cell) = self.lstm(projected)
        
        # Concatenate forward and backward final hidden state
        hidden_last = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)  # [batch_size, hidden_dim*2]
        logits = self.fc(self.dropout(hidden_last))
        
        return logits


In [ ]:
lstmModel = BiLSTM(input_dim=SIGNAL_SIZE,
                   embed_dim=EMBED_DIM,
                   hidden_dim=HIDDEN_DIM,
                   output_dim=OUTPUT_DIM,
                   num_layers=NUM_LAYERS,
                   dropout=DROPOUT)
print(lstmModel)

BiLSTM(
  (proj): Linear(in_features=1, out_features=40, bias=True)
  (lstm): LSTM(40, 90, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (fc): Linear(in_features=180, out_features=5, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)


## 5. Training

### 5.1 Bi-LSTM Training

In [ ]:
LEARNING_RATE = 1e-3
EPOTCHS = 3

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(lstmModel.parameters(), lr = LEARNING_RATE)

In [ ]:
def train_lstm_model(model, datasetloader, criterion, optimizer, device):
    model.to(device)
    model.train()
    train_loss, total_samples, total_elememts = 0, 0, 0
    correct_exact, correct_elements = 0, 0
    correct_element1, correct_element2, correct_element3, correct_element4, correct_element5 = 0,0,0,0,0

    for x_batch, y_batch in datasetloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        probs = torch.sigmoid(logits)
        preds = (probs > 0.6).float()

        exact_matches = (preds == y_batch).all(dim=1)

        correct_exact += exact_matches.sum().item()
        correct_elements += (preds == y_batch).sum().item()
        correct_element1 += (preds[:, 0] == y_batch[:, 0]).sum().item()
        correct_element2 += (preds[:, 1] == y_batch[:, 1]).sum().item()
        correct_element3 += (preds[:, 2] == y_batch[:, 2]).sum().item()
        correct_element4 += (preds[:, 3] == y_batch[:, 3]).sum().item()
        correct_element5 += (preds[:, 4] == y_batch[:, 4]).sum().item()

        total_samples += y_batch.shape[0]
        total_elememts += y_batch.numel()
        train_loss += loss.item() * y_batch.shape[0]

    avg_train_loss = train_loss / total_samples
    exact_acc = correct_exact / total_samples
    elements_acc = correct_elements / total_elememts
    

    return avg_train_loss, exact_acc, elements_acc, [correct_element1, correct_element2, correct_element3, correct_element4, correct_element5], total_samples

In [ ]:
print('Training model....')
for epoch in range(1, EPOTCHS+1):

    train_loss, exact_train_accuracy, element_wise_combined_train_acc, element_wise_each_train_acc, total_samples = train_lstm_model(lstmModel, train_dataloader, criterion, optimizer, device)

    print(f'Epoch [{epoch}/{EPOTCHS}] | Train Loss: {train_loss:.3f} | Exact Train Acc: {exact_train_accuracy*100:.3f}% | Element Wise Train Acc: {element_wise_combined_train_acc*100:.3f}%')
    print(f'''Element wise loss:
        Target1 Acc: {element_wise_each_eval_acc[0]} | Target1 Acc: {(element_wise_each_eval_acc[0]/total_samples)*100:.3f}
        Target2 Acc: {element_wise_each_eval_acc[1]} | Target1 Acc: {(element_wise_each_eval_acc[1]/total_samples)*100:.3f}
        Target3 Acc: {element_wise_each_eval_acc[2]} | Target1 Acc: {(element_wise_each_eval_acc[2]/total_samples)*100:.3f}
        Target4 Acc: {element_wise_each_eval_acc[3]} | Target1 Acc: {(element_wise_each_eval_acc[3]/total_samples)*100:.3f}
        Target5 Acc: {element_wise_each_eval_acc[4]} | Target1 Acc: {(element_wise_each_eval_acc[4]/total_samples)*100:.3f}\n''')

Training model....


Epoch [1/3] | Train Loss: 0.530 | Exact Train Acc: 7.444% | Element Wise Train Acc: 75.200%
	Element wise loss:	
Target1 Acc: 1416	
Target2 Acc: 1615	
Target3 Acc: 1492	
Target4 Acc: 844	
Target5 Acc: 1401

Epoch [2/3] | Train Loss: 0.506 | Exact Train Acc: 11.444% | Element Wise Train Acc: 75.356%
	Element wise loss:	
Target1 Acc: 1416	
Target2 Acc: 1615	
Target3 Acc: 1492	
Target4 Acc: 858	
Target5 Acc: 1401

Epoch [3/3] | Train Loss: 0.504 | Exact Train Acc: 7.833% | Element Wise Train Acc: 74.844%
	Element wise loss:	
Target1 Acc: 1416	
Target2 Acc: 1615	
Target3 Acc: 1492	
Target4 Acc: 812	
Target5 Acc: 1401



```Model score
Training model over 800 rows....
Epoch [1/5] | Train Loss: 0.494 | Exact Train Acc: 24.200% | Element Wise Train Loss: 76.540%
Element wise loss:
Target1 Acc: 795
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 535
Target5 Acc: 778

Epoch [2/5] | Train Loss: 0.495 | Exact Train Acc: 20.400% | Element Wise Train Loss: 75.980%
Element wise loss:
Target1 Acc: 795
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 506
Target5 Acc: 779

Epoch [3/5] | Train Loss: 0.491 | Exact Train Acc: 22.900% | Element Wise Train Loss: 76.140%
Element wise loss:
Target1 Acc: 795
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 514
Target5 Acc: 779

Epoch [4/5] | Train Loss: 0.489 | Exact Train Acc: 34.100% | Element Wise Train Loss: 77.660%
Element wise loss:
Target1 Acc: 796
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 587
Target5 Acc: 781

Epoch [5/5] | Train Loss: 0.502 | Exact Train Acc: 22.700% | Element Wise Train Loss: 76.400%
Element wise loss:
Target1 Acc: 796
Target2 Acc: 892
Target3 Acc: 828
Target4 Acc: 526
Target5 Acc: 778

```

## 6. Evaluation

In [ ]:
def evaluate_lstm_model(model, datasetloader, device):
    model.to(device)
    model.eval()
    eval_loss, total_samples, total_elememts = 0, 0, 0
    correct_exact, correct_elements = 0, 0
    correct_element1, correct_element2, correct_element3, correct_element4, correct_element5 = 0,0,0,0,0

    for x_batch, y_batch in datasetloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        logits = model(x_batch)
        loss = criterion(logits, y_batch)

        probs = torch.sigmoid(logits)
        preds = (probs > 0.6).float()

        exact_matches = (preds == y_batch).all(dim=1)

        correct_exact += exact_matches.sum().item()
        correct_elements += (preds == y_batch).sum().item()
        correct_element1 += (preds[:, 0] == y_batch[:, 0]).sum().item()
        correct_element2 += (preds[:, 1] == y_batch[:, 1]).sum().item()
        correct_element3 += (preds[:, 2] == y_batch[:, 2]).sum().item()
        correct_element4 += (preds[:, 3] == y_batch[:, 3]).sum().item()
        correct_element5 += (preds[:, 4] == y_batch[:, 4]).sum().item()

        total_samples += y_batch.shape[0]
        total_elememts += y_batch.numel()
        eval_loss += loss.item() * y_batch.shape[0]

    avg_eval_loss = eval_loss / total_samples
    exact_acc = correct_exact / total_samples
    elements_acc = correct_elements / total_elememts

    return avg_eval_loss, exact_acc, elements_acc, [correct_element1, correct_element2, correct_element3, correct_element4, correct_element5], total_samples

In [ ]:
print('Evaluating model....')

eval_loss, exact_eval_accuracy, element_wise_combined_eval_acc, element_wise_each_eval_acc, total_samples = evaluate_lstm_model(lstmModel, val_dataloader, device)

print(f'Epoch [{epoch}/{EPOTCHS}] | eval Loss: {eval_loss:.3f} | Exact eval Acc: {exact_eval_accuracy*100:.3f}% | Element Wise eval Acc: {element_wise_combined_eval_acc*100:.3f}%')
print(f'''Element wise loss:
      Target1 Acc: {element_wise_each_eval_acc[0]} | Target1 Acc: {element_wise_each_eval_acc[0]/total_samples}
      Target2 Acc: {element_wise_each_eval_acc[1]} | Target1 Acc: {element_wise_each_eval_acc[1]/total_samples}
      Target3 Acc: {element_wise_each_eval_acc[2]} | Target1 Acc: {element_wise_each_eval_acc[2]/total_samples}
      Target4 Acc: {element_wise_each_eval_acc[3]} | Target1 Acc: {element_wise_each_eval_acc[3]/total_samples}
      Target5 Acc: {element_wise_each_eval_acc[4]} | Target1 Acc: {element_wise_each_eval_acc[4]/total_samples}\n''')

Evaluating model....
Epoch [3/3] | eval Loss: 0.539 | Exact eval Acc: 2.038% | Element Wise eval Acc: 74.320%
	Element wise loss:
      Target1 Acc: 6249 | Target1 Acc: 0.156225
      Target2 Acc: 7055 | Target1 Acc: 0.176375
      Target3 Acc: 6227 | Target1 Acc: 0.155675
      Target4 Acc: 4095 | Target1 Acc: 0.102375
      Target5 Acc: 6102 | Target1 Acc: 0.15255



## 7. Visualization